# Bias-corrected EMA with resets via lax.scan

`jax.lax.scan` is JAX's compiled loop: it threads a **carry** through a function applied to each slice of a sequence and stacks the per-step outputs. Python `for` loops inside `jit` are unrolled, so a 10,000-step loop becomes 10,000 copies of the body; `scan` compiles the body once.

Write `ema_with_resets(x, reset, decay, debias)`:

- `x`: `(T, D)` float array, `T >= 1`.
- `reset`: `(T,)` bool. `reset[t] = True` starts a new segment at step `t` (as at an episode boundary).
- `decay`: float in `[0, 1)`; `debias`: Python bool.

Carry a state `s` (starting at zeros) and a count `c` (starting at 0). At each step `t`:

1. If `reset[t]`, set `s = 0` and `c = 0` first.
2. $s \leftarrow \text{decay}\cdot s + (1 - \text{decay})\, x_t$ and $c \leftarrow c + 1$.
3. Output $s / (1 - \text{decay}^{c})$ if `debias`, else $s$.

Return `(ema, final_s)`: the `(T, D)` stacked outputs and the raw (not bias-corrected) state after the last step, `(D,)`. float32. Use `jax.lax.scan`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def ema_with_resets(x, reset, decay, debias):
    """Return (ema (T, D), raw final state (D,)) computed with jax.lax.scan."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [false, false, false], "dtype": "bool"}, 0.5, false], "e": {"__tup__": [{"__nd__": [[0.5], [0.75], [0.875]], "dtype": "float32"}, {"__nd__": [0.875], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [false, false, false], "dtype": "bool"}, 0.5, true], "e": {"__tup__": [{"__nd__": [[1.0], [1.0], [1.0]], "dtype": "float32"}, {"__nd__": [0.875], "dtype": "float32"}]}}, {"a": [{"__nd__": [[2.0, 0.0], [4.0, 1.0], [6.0, 2.0], [8.0, 3.0]], "dtype": "float64"}, {"__nd__": [false, false, true, false], "dtype": "bool"}, 0.9, true], "e": {"__tup__": [{"__nd__": [[1.9999996423721313, 0.0], [3.052630662918091, 0.5263156294822693], [5.999999046325684, 1.9999996423721313], [7.0526299476623535, 2.526315212249756]], "dtype": "float32"}, {"__nd__": [1.340000033378601, 0.48000001907348633], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.101262453505847, 0.3384312766461778, -0.5399715152535035], [-1.2602418568524327, -1.8946212698392553, 0.018638290983285614], [-0.8105670995116028, -0.8721559599345132, -0.22196950708389104], [-0.05184602813201771, -2.2767828157758307, 0.9251465905764266], [-2.026845605910026, 1.8596241085543468, 0.5905681115998624], [-0.47184451813331413, 1.342996537380774, 0.020786461917443023]], "dtype": "float64"}, {"__nd__": [true, false, false, true, true, false], "dtype": "bool"}, 0.0, true], "e": {"__tup__": [{"__nd__": [[1.1012624502182007, 0.3384312689304352, -0.5399715304374695], [-1.2602418661117554, -1.8946212530136108, 0.018638290464878082], [-0.8105670809745789, -0.8721559643745422, -0.22196950018405914], [-0.05184602737426758, -2.276782751083374, 0.9251465797424316], [-2.026845693588257, 1.8596241474151611, 0.5905681252479553], [-0.47184452414512634, 1.3429964780807495, 0.020786462351679802]], "dtype": "float32"}, {"__nd__": [-0.47184452414512634, 1.3429964780807495, 0.020786462351679802], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.6909209676232841, 0.10722707944720507, 1.096531380130275, 1.0614224543758295], [-0.9073341413930657, -0.6124917784436158, 0.3432270521124315, -0.21001015163517062], [-2.284914216554451, 2.0261175349217764, -2.174977373868806, -2.081542853744764], [-1.2761124197329383, 0.5648512404463458, 1.7960125066302708, -0.2354150525900867], [-0.3910773505445041, 0.17390146428778722, -0.3668597432540227, 0.08236522460221347], [-1.1399714593630075, 0.438257432619766, 0.39257939815601534, -2.4354174163439515], [-0.11022594532382894, 0.16291581767584848, 0.5612247602630475, -0.816089756439701], [-1.3280342642624956, -0.9507340805091575, 0.5480409641909273, -0.8518355089919037], [-0.7641438115235788, 1.1437655777004572, -0.7861575894791125, -0.8986326207536771], [-0.09123756351337108, 0.19861051640677327, 1.0376296586712488, -0.8793276811910162]], "dtype": "float64"}, {"__nd__": [false, false, false, false, false, false, false, false, false, false], "dtype": "bool"}, 0.99, true], "e": {"__tup__": [{"__nd__": [[0.6909216046333313, 0.10722718387842178, 1.0965324640274048, 1.0614235401153564], [-0.11222251504659653, -0.25444114208221436, 0.7179877758026123, 0.42251238226890564], [-0.8437439203262329, 0.513398289680481, -0.25604262948036194, -0.4205760657787323], [-0.953471302986145, 0.5264562368392944, 0.26473096013069153, -0.3735857307910919], [-0.8387206196784973, 0.4545210003852844, 0.1358610838651657, -0.2805534601211548], [-0.8901996612548828, 0.45174214243888855, 0.17972981929779053, -0.648781955242157], [-0.7753872871398926, 0.40922680497169495, 0.23588605225086212, -0.6734098196029663], [-0.8469222187995911, 0.23319195210933685, 0.2762916386127472, -0.6965052485466003], [-0.8373510837554932, 0.3384818732738495, 0.1534406691789627, -0.7198777198791504], [-0.7593201994895935, 0.32385364174842834, 0.24591179192066193, -0.7365533709526062]], "dtype": "float32"}, {"__nd__": [-0.07260455191135406, 0.03096618317067623, 0.0235135518014431, -0.07042763382196426], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.5681545944352957, -0.12249665156557239, 0.6436368157294202, -1.481526543249413], [-0.48726301578469694, 0.30619578810330356, 0.2310243000221297, 0.08335899091739198], [-0.2568895173775524, -0.7993464580715008, -0.9281380009611975, 1.1562622342720783], [-1.1172097148854832, 1.2280000985175472, -1.3150584358653277, 2.903581786972847], [-1.7381109288833918, 1.6042742994540782, -0.8970721577902043, -0.11561548050669124], [-0.06555257091474957, -0.6522226650767547, 2.2283926348223857, 0.09491534615827704], [-1.3210737429953536, 0.6465603079512943, 0.7124017466131326, -1.1474830383225882], [0.5356130419751398, 0.06895578792877383, 0.8203356602681026, 0.7121954717945739], [-0.008442406578893256, -0.4082828680274049, 0.9103726831466016, 0.3131992040260024], [-1.4892773731328215, -0.5448691737581977, -0.4833123986084571, -0.2621545059158811]], "dtype": "float64"}, {"__nd__": [false, true, false, false, false, false, false, true, false, false], "dtype": "bool"}, 0.8, false], "e": {"__tup__": [{"__nd__": [[-0.3136309087276459, -0.024499330669641495, 0.12872736155986786, -0.2963052988052368], [-0.09745260328054428, 0.06123916059732437, 0.046204861253499985, 0.01667179726064205], [-0.12933999300003052, -0.11087796837091446, -0.1486637145280838, 0.2445898950099945], [-0.3269139528274536, 0.156897634267807, -0.38194265961647034, 0.7763882875442505], [-0.6091533303260803, 0.44637298583984375, -0.4849685728549957, 0.5979875326156616], [-0.5004332065582275, 0.22665385901927948, 0.057703666388988495, 0.497373104095459], [-0.6645613312721252, 0.31063514947891235, 0.18864329159259796, 0.16840189695358276], [0.10712261497974396, 0.013791157864034176, 0.1640671342611313, 0.1424390971660614], [0.08400961011648178, -0.07062365114688873, 0.31332823634147644, 0.17659111320972443], [-0.23064777255058289, -0.16547276079654694, 0.15400010347366333, 0.0888419896364212]], "dtype": "float32"}, {"__nd__": [-0.23064777255058289, -0.16547276079654694, 0.15400010347366333, 0.0888419896364212], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.4120768860796877, 2.319642474365703], [1.834109164605156, 2.8866275042889975], [4.3980120333855055, 2.7735842440011385], [4.148429491914847, 3.4125987091203513], [2.691102969961726, 0.36660508783100454], [3.222133655959293, 2.723054858836126], [1.399480743209398, 3.5145047983461843], [2.974577482345076, 3.3750530708216506], [2.395584104119201, 3.2020225611372157], [2.526768063964152, 4.064946673093174], [3.1243256664769268, 1.5974757182561437], [3.2143160472170793, 3.0676411152902054], [2.7965556009256733, 1.6503377533850694], [4.059537780127499, 2.2684030800434076], [4.582635441947494, 2.3400560178189247], [3.032476859354181, 2.955187079557843], [3.1511744275997167, 5.64341501004198], [2.5017121955293975, 5.06945735394623], [3.623458224174904, 2.3049924589511743], [4.255560460885063, 3.840824406337222], [3.954436851995186, 1.6660155980753322], [3.5028116051024147, 2.597484103531394], [2.8578344007396206, 2.868541716522012], [3.0406683473704565, 4.590443538676863], [3.378440191647275, 1.155238962546793], [4.246867713732077, 3.0876431062446343], [3.1315273825358414, 1.1733399671493066], [3.3997533228551635, 2.5010345321401197], [3.9556199583269502, 3.4562282140102334], [3.827719278221387, 2.6269481891236097], [1.1706512794742492, 3.968792693159547], [3.4203499347269197, 2.669380205757956], [2.442935584831853, 2.114388657581424], [1.2374633968319706, 1.8227963242069798], [2.9298877538910935, 4.609464007293136], [3.453165140804062, 4.604778779301444], [1.9901768221776415, 4.2566816536451375], [2.5838668509826537, 4.466329218886482], [4.228395170927101, 3.0353628047166064], [3.1344969966661957, 4.217434767651174], [2.2042356156120655, 0.8370891132641178], [3.816929662887744, 5.17912168814974], [1.1038164643674286, 3.209642506963478], [1.8178456736796273, 2.709979931601828], [1.9999758592753494, 3.874921106279234], [2.4548159838248207, 2.807152364800011], [1.9162300379690147, 3.7328925977842218], [2.490126290687507, 2.6216254969895743], [2.9244017307631602, 1.5482372120339412], [2.9791574473836073, 1.3881042551552738]], "dtype": "float64"}, {"__nd__": [false, false, false, false, false, true, false, false, false, false, false, true, false, false, false, false, false, false, false, false, false, false, false, true, false, false, false, false, false, false, false, false, false, false, false, true, false, false, false, false, false, false, false, false, false, false, false, false, false, false], "dtype": "bool"}, 0.95, true], "e": {"__tup__": [{"__nd__": [[3.412076473236084, 2.3196420669555664], [2.6028621196746826, 2.610403537750244], [3.232186794281006, 2.667609453201294], [3.479160785675049, 2.868422031402588], [3.3049798011779785, 2.315458297729492], [3.222133159637451, 2.7230544090270996], [2.2874393463134766, 3.1289260387420654], [2.5283288955688477, 3.215210437774658], [2.4925475120544434, 3.211655855178833], [2.500110626220703, 3.4002535343170166], [2.6179282665252686, 3.0599894523620605], [3.214315414428711, 3.0676403045654297], [3.0000791549682617, 2.340818166732788], [3.371492862701416, 2.3154313564300537], [3.697957754135132, 2.3220691680908203], [3.5508692264556885, 2.462003231048584], [3.4754295349121094, 3.0624780654907227], [3.314037561416626, 3.3951308727264404], [3.3600029945373535, 3.233187198638916], [3.4811055660247803, 3.3153555393218994], [3.540085792541504, 3.109837055206299], [3.5357635021209717, 3.050426959991455], [3.462017774581909, 3.0306410789489746], [3.040667772293091, 4.590442657470703], [3.213883876800537, 2.8287994861602783], [3.5760161876678467, 2.919541597366333], [3.4562041759490967, 2.4488518238067627], [3.4437263011932373, 2.4603850841522217], [3.540343999862671, 2.648345470428467], [3.5879757404327393, 2.644798755645752], [3.228874444961548, 2.841482400894165], [3.2547667026519775, 2.818209171295166], [3.153607130050659, 2.73050856590271], [2.931419849395752, 2.6252543926239014], [2.931252956390381, 2.8410983085632324], [3.4531643390655518, 4.604777812957764], [2.702913999557495, 4.426266193389893], [2.661179304122925, 4.440310478210449], [3.0836236476898193, 4.061605930328369], [3.09486722946167, 4.096047401428223], [2.9267656803131104, 3.4809370040893555], [3.0743086338043213, 3.7624075412750244], [2.781585454940796, 3.6802923679351807], [2.6512622833251953, 3.5490803718566895], [2.5701076984405518, 3.5896823406219482], [2.556739091873169, 3.498943567276001], [2.4870636463165283, 3.524392604827881], [2.4873783588409424, 3.431640863418579], [2.530029058456421, 3.247831106185913], [2.5718703269958496, 3.0745785236358643]], "dtype": "float32"}, {"__nd__": [1.3803454637527466, 1.650153398513794], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(ema_with_resets, json.loads(_TESTS), 'jax', *[0.0001, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")